# Dimensión Novedad (dim_novedad) - Fast and Safe ETL

# Imports

In [ ]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import String, create_engine, text
import hashlib

In [ ]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

config_co = config["CO_SA"]
config_etl = config["ETL_PRO"]

co_sa = create_engine(
    f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:{config_co['port']}/{config_co['dbname']}"
)

etl_conn = create_engine(
    f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:{config_etl['port']}/{config_etl['dbname']}"
)

# Leer tabla origen

df_tipo = pd.read_sql_table(
    "mensajeria_tiponovedad",
    con=co_sa
)

print(df_tipo)

In [ ]:
df = pd.read_sql_table("mensajeria_tiponovedad", con=co_sa)

print(df.columns.tolist())

display(df.head())

## Transformacion

In [ ]:
# Construcción de dim_novedad

dim_novedad = pd.DataFrame()

# Llave sustituta
dim_novedad["key_dim_novedad"] = range(1, len(df_tipo) + 1)

# Llave de negocio
dim_novedad["id_novedad_origen"] = df_tipo["id"]

# Nombre del tipo de novedad
dim_novedad["categoria_novedad"] = df_tipo["nombre"]

# Como no existe descripción, reutilizamos el nombre
dim_novedad["descripcion"] = df_tipo["nombre"]

display(dim_novedad)

## Validaciones

In [ ]:
print(dim_novedad.shape)
print(dim_novedad)

print(
    dim_novedad["id_novedad_origen"].duplicated().sum()
)

## Creamos la tabla

In [ ]:
sql = """
DROP TABLE IF EXISTS dim_novedad;

CREATE TABLE dim_novedad(

    key_dim_novedad INTEGER PRIMARY KEY,

    id_novedad_origen INTEGER NOT NULL,

    categoria_novedad VARCHAR(150),

    descripcion VARCHAR(255)

);
"""

with etl_conn.begin() as conn:
    conn.execute(text(sql))

print("Tabla creada correctamente.")

## Carga

In [ ]:
dim_novedad.to_sql(
    "dim_novedad",
    con=etl_conn,
    if_exists="append",
    index=False
)

print("Dimensión cargada.")